### 🎯 [미션] 주석이 달린 줄의 None 또는 _______ 부분을 채워 코드를 완성하세요.

각 셀을 순서대로 실행(Shift + Enter)해야 합니다.

---

# 🧪 안전모 탐지 모델 평가

이 노트북에서는 학습된 모델의 성능을 테스트 데이터셋으로 평가합니다.

**평가 지표:**
- **Precision (정밀도)**: 모델이 탐지한 것 중 실제로 맞은 비율
- **Recall (재현율)**: 실제 객체 중 모델이 탐지한 비율
- **mAP50**: IoU 50% 기준 평균 정밀도
- **mAP50-95**: IoU 50~95% 기준 평균 정밀도

---
#### 1️⃣ 필요한 라이브러리 설치 및 불러오기

In [ ]:
# YOLO 라이브러리 설치 (최초 1회만 실행)
!pip install ultralytics

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO
from ultralytics import settings

print("✅ 라이브러리 로딩 완료!")

---
#### 2️⃣ 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/2026_AI_Advanced_Study-main/4차시/04_hardhat_detection/code"

In [ ]:
# 현재 작업 디렉토리 확인
current_dir = os.getcwd()
new_runs_dir = os.path.abspath(os.path.join(current_dir, '../runs'))
new_data_dir = os.path.abspath(os.path.join(current_dir, '../data'))

# YOLO 설정 업데이트
settings.update({"runs_dir": new_runs_dir})
settings.update({"datasets_dir": new_data_dir})
settings.update({"wandb": False})

print(f"✅ 작업 디렉토리: {current_dir}")
print("\n✅ 경로 설정 완료!")

---
#### 3️⃣ 학습된 모델 불러오기

In [ ]:
# 🎯 [미션] 학습된 모델의 경로를 설정하세요.
MODEL_PATH = '../runs/detect/train/weights/best.pt'

# 모델 로드
model = YOLO(MODEL_PATH)

print(f"✅ 모델 로드 완료: {MODEL_PATH}")

---
#### 4️⃣ 테스트 데이터셋으로 모델 평가

In [ ]:
# 데이터 설정 파일 경로
DATA_CONFIG = '../data/config.yaml'

print("🧪 테스트 데이터셋으로 모델 평가 중...")

# 🎯 [미션] 모델 검증을 수행하는 메서드를 호출하세요.
# 힌트: 학습은 train(), 검증은?
results = model.val(
    data=DATA_CONFIG,
    split='test'  # test 데이터셋 사용
)

print("\n✅ 평가 완료!")

---
#### 5️⃣ 평가 결과 해석

Detection 모델의 주요 평가 지표:
- **mAP50**: IoU 50% 이상일 때의 평균 정밀도
- **mAP50-95**: IoU 50%~95%의 평균 정밀도 (더 엄격한 기준)

In [ ]:
print("=" * 50)
print("📊 평가 결과 요약")
print("=" * 50)

# 클래스별 결과
class_names = model.names
print(f"\n📌 클래스: {class_names}")

print(f"\n🎯 전체 성능:")
print(f"   - mAP50: {results.box.map50:.4f}")
print(f"   - mAP50-95: {results.box.map:.4f}")

---
#### 6️⃣ 샘플 이미지 예측 테스트

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob

# 테스트 이미지 가져오기
test_images = glob.glob('../data/test/images/*.[jJ][pP][gG]')[:3]

print(f"📷 테스트할 이미지: {len(test_images)}장\n")

for img_path in test_images:
    print(f"\n🖼️ 이미지: {os.path.basename(img_path)}")
    
    # 예측 수행
    results = model.predict(img_path, verbose=False)
    
    # 🎯 [미션] Detection 결과에서 바운딩 박스 정보를 가져오세요.
    # 힌트: Classification은 probs, Detection은?
    boxes = results[0].boxes
    
    print(f"   탐지된 객체 수: {len(boxes)}개")
    
    # 클래스별 개수 출력
    for box in boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        cls_name = model.names[cls_id]
        print(f"   - {cls_name}: {conf:.2%}")
    
    # 결과 시각화
    result_img = results[0].plot()
    plt.figure(figsize=(10, 8))
    plt.imshow(result_img[:, :, ::-1])  # BGR to RGB
    plt.axis('off')
    plt.title(f"Detected: {len(boxes)} objects")
    plt.show()